# Session 2, part 1: Objects and classes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-hossein/hands-on-ml-biotech/blob/main/02_ObjectsAndData/01_OOP.ipynb)

Last session we wrote functions. Today we look at the other building block of Python code: **classes**. Every model you will use in this course (`LinearRegression`, a PyTorch network) is a class, so you need to be able to read one, and it helps to have written a few.


---
## 1. Two ways to organise code

- **Functions transform data.** You give data in and get a result back. The function remembers nothing.

- **Objects hold state.** An object keeps its data and the functions that work on it together.

In [ ]:
# Function style: the data is passed in every time
def gc_content(seq):
    return (seq.count("G") + seq.count("C")) / len(seq)

seq = "ATGGAGGAGCCGCAGTCAGATCC"
print(gc_content(seq))

Both styles appear in this course:

- NumPy and pandas operations are mostly **function style**: `X.mean()` gives you a new value and leaves `X` alone.
- A machine learning model is an **object** (or an **instance**): after `model.fit(X, y)` it remembers what it learned, and `model.predict(X_new)` uses that memory.

As per python documentation: *Classes provide a means of bundling data and functionality together*

When something needs to remember things between calls, use a class.

## 2. You have used objects all along

In Python everything is an object.

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/02_ObjectsAndData/images/01_OOP_cell3_image.png)

*from https://www.geeksforgeeks.org/python/python-classes-and-objects/*

Its class (its type) decides which data it holds and which functions it has. A function that belongs to an object is called a **method**. 

Similarly, a variable that belongs to an object is called its **attribute** (called **field** in C++). Attributes can belong to an object (Instance attribute, e.g., self.name) or shared by all objects from that class (class attribute).

Furthermore, when you're building an object, you pass arguments to its **constructor** through the `__init__` function. Constructors

In [ ]:
genes = ["TP53", "BRCA1"]
print(type(genes))
genes.append("MYC")
print(genes)

In [ ]:
# str and int, isinstance

## 3. Writing a class

- `class` defines a new type.
- `__init__` runs when you create an object. It stores the data.
- `self` is the object itself. Data stored on it (`self.name`) is called an **attribute**.
- Functions defined inside the class are its **methods**. Their first argument is always `self`.

In [ ]:
class DNASequence:
    def __init__(self, name, seq): # Constructor
        self.name = name # Attributes: the data this object holds
        self.seq = seq.upper()

    def gc_content(self): # Methods: uses the object's own data
        return (self.seq.count("G") + self.seq.count("C")) / len(self.seq)

In [ ]:
# One class, many objects (also called instances)
tp53 = DNASequence("TP53", "atggaggagccgcagtcagatcc")
myc = DNASequence("MYC", "ATGCCCCTCAACGTTAGCTTC")


`tp53.gc_content()` is short for `DNASequence.gc_content(tp53)`. Python passes the object in as `self` for you. That is all `self` is.

In [ ]:
print(DNASequence.gc_content(tp53) == tp53.gc_content())

### ✏️ Write a class `Sample` that stores a `name` and a list of expression `values`, and has a method `mean()` that returns the mean of the values.

In [ ]:
class Sample:
    # YOUR CODE HERE
    pass

## 4. Dunder methods

Try printing our object, or asking for its length:

In [ ]:
print(tp53)

In [ ]:
len(tp53)

Python's built-in operations look for methods with special names. They start and end with a **d**ouble **under**score, so people call them *dunder* methods. You never call them yourself. Python calls them for you:

| You write | Python calls |
| --- | --- |
| `DNASequence(...)` | `__init__` |
| `print(obj)` or `obj` at the end of a cell | `__repr__` |
| `len(obj)` | `__len__` |
| `obj[i]` | `__getitem__` |
| `a + b` | `__add__` |
| `a == b` | `__eq__` |
| `obj(x)` | `__call__` |

In [ ]:
class DNASequence:
    def __init__(self, name, seq):
        self.name = name
        self.seq = seq.upper()

    def gc_content(self):
        return (self.seq.count("G") + self.seq.count("C")) / len(self.seq)

    def __repr__(self): # print(obj)
        return f"DNASequence({self.name}, {len(self.seq)} bp)"

    def __len__(self): # len(obj)
        return len(self.seq)

    def __getitem__(self, index): # obj[index]
        return self.seq[index]

    def __add__(self, other): # obj + other
        return DNASequence(self.name + "+" + other.name, self.seq + other.seq)

    def __eq__(self, other): # obj == other
        return self.seq == other.seq

In [ ]:
tp53 = DNASequence("TP53", "atggaggagccgcagtcagatcc")
tpp3_copy = DNASequence("copy", "ATGGAGGAGCCGCAGTCAGATCC")
myc = DNASequence("MYC", "ATGCCCCTCAACGTTAGCTTC")

# try print, len, getitem, add, and eq


`__call__` lets you use an object like a function. 

PyTorch models work this way: you write `model(x)`.

In [ ]:
class Scaler:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return x * self.factor

double = Scaler(2)
print(double(5)) # looks like a function call
print(double(np.array([1, 2, 3])))

### ✏️ PyTorch expects a dataset to be an object that answers two questions: how many samples do you have (`len(dataset)`), and what is sample number `i` (`dataset[i]`).

Complete `ExpressionDataset` so that `len(dataset)` gives the number of samples and `dataset[i]` gives the pair `(X[i], y[i])`.

In [ ]:
class ExpressionDataset:
    def __init__(self, X, y):
        self.X = X
        self.y = y

    # YOUR CODE HERE

## 5. Inheritance

A class can be built **on top of** another class. The new class (the **child**, also called the **subclass**) gets everything the **parent** has, and can add or change things.

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/02_ObjectsAndData/images/01_OOP_cell23_image.png)

*from https://python.land/objects-and-classes/python-inheritance, This is called a **UML** representation*

A gene is a DNA sequence with some extra information. So we don't start from zero:

In [ ]:
class Gene(DNASequence):
    def __init__(self, name, seq, chromosome):
        super().__init__(name, seq) 
        self.chromosome = chromosome # new attributes

    def __repr__(self): # replace the parent's version (this is call overriding)
        return f"Gene({self.name}, chr{self.chromosome}, {len(self)} bp)"

    def location(self): # a new method
        return f"{self.name} is on chromosome {self.chromosome}"

In [ ]:
tp53_gene = Gene("TP53", "atggaggagccgcagtcagatcc", chromosome=17)

# now try print, gc_content, and len




In [ ]:
print(isinstance(tp53_gene, Gene), isinstance(tp53_gene, DNASequence))

## 6. How Python finds a method (MRO)

When you call `tp53_gene.gc_content()`, Python looks for `gc_content` in the object's own class first, then in its parent, then in the parent's parent, and uses the **first one it finds**. This search order is the **method resolution order (MRO)**.

That is why overriding works: `Gene.__repr__` is found before `DNASequence.__repr__`.

In [ ]:
print(Gene.__mro__)

`object` is at the end of every chain. It is the parent of all classes, and it is where the unhelpful default `__repr__` from earlier came from.

In this course a class has one parent at most, so the MRO is always a simple chain.

## 7. Putting it together: a tiny model

Every scikit-learn model follows the same pattern:

- `fit(X, y)` learns from data and **stores** what it learned on the object.
- `predict(X)` uses what was stored.

Let's write the simplest possible model: it always predicts the mean of the training targets.

In [ ]:
class MeanPredictor:
    def fit(self, X, y):
        self.constant_ = np.mean(y) # learned values end with an underscore
        return self

    def predict(self, X):
        return np.full(len(X), self.constant_)

In [ ]:
X_train = np.array([[1.0], [2.0], [3.0], [4.0]])
y_train = np.array([10.0, 12.0, 15.0, 19.0])
X_new = np.array([[5.0], [6.0]])


### ✏️ Predict the y values for X_new using MeanPredictor